# Broadcast homography invalid intervals

This notebook maps broadcast frames whose homography is marked invalid onto the tactical-video timeline, then saves each contiguous invalid interval to a CSV. It uses a one-to-one frame-rate mapping from the supplied half-start anchors. The result is only as accurate as those anchors and assumes no clock drift within each half; it does not determine whether an invalid segment is definitely a replay.

Run the cells from top to bottom. The broadcast homography CSV is loaded through the repository's `games.asset_paths` helper.

In [ ]:
# Editable frame anchors: first frame of each half in the respective video.
BROADCAST_FIRST_HALF_START = 7150
BROADCAST_SECOND_HALF_START = 86825
TACTICAL_FIRST_HALF_START = 11185
TACTICAL_SECOND_HALF_START = 113742
FPS = 25

OUTPUT_CSV = "broadcast_invalid_homography_tactical_timeline.csv"

from pathlib import Path

import pandas as pd

from games import asset_paths

homography_path = Path(asset_paths(source="broadcast")["homography"])
print(f"Homography input: {homography_path}")
print(f"CSV output:       {Path.cwd() / OUTPUT_CSV}")

In [ ]:
if not homography_path.is_file():
    raise FileNotFoundError(
        f"Broadcast homography CSV not found: {homography_path}. "
        "Check that the match data is in the repository's data/broadcast folder "
        "or configure MTAH_DATA_DIR before starting the notebook kernel."
    )

homography = pd.read_csv(homography_path)
homography.columns = homography.columns.str.strip().str.lower()
required_columns = {"frame", "valid"}
missing_columns = required_columns - set(homography.columns)
if missing_columns:
    raise ValueError(f"Homography CSV is missing required columns: {sorted(missing_columns)}")

homography["frame"] = pd.to_numeric(homography["frame"], errors="raise").astype(int)
homography["valid"] = pd.to_numeric(homography["valid"], errors="raise")
if homography["frame"].duplicated().any():
    raise ValueError("Homography CSV contains duplicate frame numbers.")
if homography["valid"].isna().any():
    raise ValueError("Homography CSV contains missing validity values.")

homography = homography.sort_values("frame").reset_index(drop=True)
second_half_broadcast_start = BROADCAST_SECOND_HALF_START
half_anchors = [
    ("first", BROADCAST_FIRST_HALF_START, second_half_broadcast_start, TACTICAL_FIRST_HALF_START),
    ("second", second_half_broadcast_start, None, TACTICAL_SECOND_HALF_START),
]

mapped_parts = []
for half, broadcast_start, broadcast_end, tactical_start in half_anchors:
    in_half = homography["frame"] >= broadcast_start
    if broadcast_end is not None:
        in_half &= homography["frame"] < broadcast_end
    part = homography.loc[in_half, ["frame", "valid"]].copy()
    part["half"] = half
    part["tactical_frame"] = tactical_start + part["frame"] - broadcast_start
    part["invalid"] = part["valid"].eq(0)
    mapped_parts.append(part)

mapped = pd.concat(mapped_parts, ignore_index=True).sort_values("frame").reset_index(drop=True)
if mapped.empty:
    raise ValueError("No homography rows fall on or after the configured half-start frames.")

print(f"Mapped {len(mapped):,} broadcast homography frames.")
print(f"Invalid frames: {int(mapped['invalid'].sum()):,}")
mapped.head()

In [ ]:
def frame_timestamp(frame):
    """Format a video-frame index as HH:MM:SS.mmm from that video's frame zero."""
    milliseconds = round(int(frame) * 1000 / FPS)
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    seconds, milliseconds = divmod(remainder, 1000)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}.{milliseconds:03d}"


# Start a new interval when validity changes, a homography frame is missing, or the half changes.
new_run = (
    mapped["half"].ne(mapped["half"].shift())
    | mapped["invalid"].ne(mapped["invalid"].shift())
    | mapped["frame"].diff().ne(1)
)
mapped["run"] = new_run.cumsum()

intervals = (
    mapped.loc[mapped["invalid"]]
    .groupby(["run", "half"], sort=True)
    .agg(
        broadcast_start_frame=("frame", "min"),
        broadcast_end_frame=("frame", "max"),
        tactical_start_frame=("tactical_frame", "min"),
        tactical_end_frame=("tactical_frame", "max"),
    )
    .reset_index()
    .drop(columns="run")
)

intervals["tactical_start_time"] = intervals["tactical_start_frame"].map(frame_timestamp)
intervals["tactical_end_time"] = intervals["tactical_end_frame"].map(frame_timestamp)
intervals["invalid_duration_seconds"] = (
    intervals["tactical_end_frame"] - intervals["tactical_start_frame"] + 1
) / FPS

columns = [
    "half",
    "broadcast_start_frame",
    "broadcast_end_frame",
    "tactical_start_frame",
    "tactical_end_frame",
    "tactical_start_time",
    "tactical_end_time",
    "invalid_duration_seconds",
]
intervals = intervals[columns]
intervals.to_csv(OUTPUT_CSV, index=False)

print(f"Found {len(intervals):,} contiguous invalid intervals.")
print(f"Saved: {Path(OUTPUT_CSV).resolve()}")
intervals